In [ ]:
# ============================================================
# CELL 0 – LOCAL ENVIRONMENT SETUP
# ============================================================
import os, sys

# Cấu trúc Local-first: lấy thư mục gốc của project
# Vì notebook nằm trong thư mục notebooks/, nên PROJECT_ROOT là thư mục cha.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
REPO_PATH = PROJECT_ROOT

# Add src/ to Python path
if os.path.join(PROJECT_ROOT, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

print('✅ Local Environment ready. PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
import os
import subprocess
import threading
import time
import sys

# Set model source
os.environ["HF_MODEL_ID"] = "thong7d/vihsd-xlmr-hate-speech"

# Install ngrok
os.system("pip install -q pyngrok")
from pyngrok import ngrok

# Cấu hình Authtoken
ngrok.set_auth_token("3CO8IpvERS1b7tqBq7BC9lygnyP_3b93zW7mEhaJWWCycRUEo")

# Khai báo biến REPO_PATH trỏ về thư mục làm việc hiện tại (hoặc thay bằng đường dẫn cụ thể)
REPO_PATH = os.getcwd()

# Start FastAPI in background thread
def run_uvicorn():
    import uvicorn
    # Import app to trigger model loading
    sys.path.insert(0, f"{REPO_PATH}/src")
    from api.app import app
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_uvicorn, daemon=True)
server_thread.start()
time.sleep(10)  # Wait for model to load

# Create ngrok tunnel
public_url = ngrok.connect(8000)
print(f"✅ FastAPI is live!")
print(f"   Public URL:  {public_url}")
print(f"   Docs:        {public_url}/docs")
print(f"   Health:      {public_url}/health")

In [ ]:
import requests

# Điền đúng Public URL của ngrok (bắt buộc có https://)
API_URL = "https://uncork-bubbly-stiffly.ngrok-free.dev"

test_cases = [
    "Bài viết rất hay, cảm ơn tác giả!",
    "Đồ ngu, mày biết gì mà nói",
    "Loại người như mày không xứng đáng sống",
]

print("📊 API Test Results:")
print("=" * 70)
for text in test_cases:
    response = requests.post(f"{API_URL}/predict", json={"text": text})
    result = response.json()
    print(f"Text:       {result['text'][:60]}...")
    print(f"Label:      {result['label']} (conf: {result['confidence']:.3f})")
    print(f"Scores:     {result['scores']}")
    print(f"Borderline: {result['is_borderline']}")
    print(f"Latency:    {result['latency_ms']:.1f}ms")
    print("-" * 70)

In [ ]:
import gradio as gr
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

HF_MODEL_ID = "thong7d/vihsd-xlmr-hate-speech"  # ← Update
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(HF_MODEL_ID)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

LABEL_MAP = {0: "CLEAN ✅", 1: "OFFENSIVE ⚠️", 2: "HATE 🚫"}
LABEL_COLORS = {0: "#4CAF50", 1: "#FF9800", 2: "#F44336"}

def predict_text(text):
    """Predict hate speech label for input text."""
    if not text or len(text.strip()) < 3:
        return "Please enter at least 3 characters.", "", ""

    encoding = tokenizer(
        text, max_length=128, padding="max_length",
        truncation=True, return_tensors="pt"
    )
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]

    pred_id = int(probs.argmax())
    confidence = float(probs.max())

    label_text = LABEL_MAP[pred_id]
    scores_text = "\n".join([
        f"CLEAN:     {probs[0]:.4f}",
        f"OFFENSIVE: {probs[1]:.4f}",
        f"HATE:      {probs[2]:.4f}",
    ])
    conf_text = f"{confidence:.2%}"

    return label_text, conf_text, scores_text

demo = gr.Interface(
    fn=predict_text,
    inputs=gr.Textbox(
        label="Enter Vietnamese text",
        placeholder="Nhập văn bản tiếng Việt tại đây...",
        lines=3,
    ),
    outputs=[
        gr.Textbox(label="Prediction"),
        gr.Textbox(label="Confidence"),
        gr.Textbox(label="All Scores"),
    ],
    title="🛡️ Vietnamese Hate Speech Detector",
    description=(
        "Powered by XLM-RoBERTa fine-tuned on the ViHSD dataset (~33K comments). "
        "Classifies text as CLEAN, OFFENSIVE, or HATE."
    ),
    examples=[
        ["Bài viết rất hay, cảm ơn tác giả!"],
        ["Mày ngu quá, biết gì mà nói"],
        ["Loại người như mày không xứng đáng sống trên đời này"],
    ],
    theme=gr.themes.Soft(),
    allow_flagging="never",
)

demo.launch(share=True)